In [ ]:
"""
  RAINFALL ANALYSIS — CITIES (8) & STATES (9)  |  2005-2017
  Data sources:
    • yearly_mean_rainfall_builtup_cities.csv 
    • yearly_mean_rainfall_builtup_states.csv  
    • jjas_mean_rain_cities.csv               
    • jjas_mean_rain_states.csv                

  Section map:
    1.  Setup — imports, global style, palettes
    2.  Data loading & helper functions
    3.  Normal distribution fitting   (yearly | cities & states)
    4.  Gamma distribution fitting    (yearly | cities & states)
    5.  Data smoothing — 3-yr & 5-yr MA (yearly | cities & states)
    6.  Time series                   (JJAS   | cities & states)
    7.  Hypothesis testing            (JJAS    | cities & states)
"""

# SECTION 1 — IMPORTS, GLOBAL STYLE, PALETTES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import norm, gamma, kstest, ttest_ind, mannwhitneyu

In [ ]:
# Global matplotlib style 
plt.rcParams.update({
    "figure.facecolor"  : "#ffffff",
    "axes.facecolor"    : "#f4f6f9",
    "axes.edgecolor"    : "#000000",
    "axes.labelcolor"   : "#000000",
    "axes.titlecolor"   : "#000000",
    "xtick.color"       : "#2c3e50",
    "ytick.color"       : "#2c3e50",
    "xtick.labelsize"   : 8,         
    "ytick.labelsize"   : 8,        
    "grid.color"        : "#c8d0db",
    "grid.linewidth"    : 0.5,
    "text.color"        : "#000000",
    "legend.facecolor"  : "#ffffff",
    "legend.edgecolor"  : "#000000",
    "legend.labelcolor" : "#000000",
    "legend.fontsize"   : 7,         
    "font.family"       : "DejaVu Sans",
    "axes.labelweight"  : "bold",
    "axes.titleweight"  : "bold",
    "axes.labelsize"    : 9,          
    "axes.titlesize"    : 10,        
})

In [ ]:
# Colour palettes 

CITY_PALETTE = [
    "#1565C0",   # vivid royal blue      (Ahmadabad)
    "#E53935",   # bold red              (Bengaluru)
    "#2E7D32",   # rich forest green     (Hyderabad)
    "#6A1B9A",   # strong purple         (Indore)
    "#EF6C00",   # vivid orange          (Jaipur)
    "#00838F",   # deep cyan-teal        (Nagpur)
    "#C62828",   # deep crimson          (Pimpri Chinchwad) 
    "#AD1457",   # hot pink-magenta      (Pune)
]

In [ ]:
STATE_PALETTE = [
    "#1565C0",   # vivid royal blue      (Andhra Pradesh)
    "#E53935",   # bold red              (Bihar)
    "#2E7D32",   # rich green            (Chhattisgarh)
    "#6A1B9A",   # strong purple         (Himachal Pradesh)
    "#EF6C00",   # vivid orange          (Jammu & Kashmir)
    "#00838F",   # deep teal             (Madhya Pradesh)
    "#AD1457",   # hot pink              (Meghalaya)
    "#558B2F",   # olive green           (Tamil Nadu)
    "#4E342E",   # warm dark brown       (Telangana)
    "#0277BD",   # bright sky blue       (Uttar Pradesh)
    "#00695C",   # deep emerald          (Uttarakhand)
]

In [ ]:
NORM_COLOR  = "#1565C0"   # vivid blue  — Normal fit line   
GAMMA_COLOR = "#E53935"   # vivid red   — Gamma  fit line 
MA3_COLOR   = "#EF6C00"   # vivid amber — 3-yr MA          
MA5_COLOR   = "#2E7D32"   # rich green  — 5-yr MA         
HIST_ALPHA  = 0.55

# SECTION 2 — DATA LOADING & HELPERS

In [ ]:
yearly_cities_df = pd.read_csv("yearly_mean_rainfall_builtup_cities.csv")
yearly_states_df = pd.read_csv("yearly_mean_rainfall_builtup_states.csv")
jjas_cities_df   = pd.read_csv("jjas_mean_rain_cities.csv")
jjas_states_df   = pd.read_csv("jjas_mean_rain_states.csv")

In [ ]:
CITIES = list(dict.fromkeys(yearly_cities_df["region"].tolist()))
STATES = list(dict.fromkeys(yearly_states_df["region"].tolist()))

In [ ]:
def get_series(df, region):
    sub   = df[df["region"] == region].sort_values("year")
    years = sub["year"].values.astype(float)
    rain  = sub["mean_rainfall"].values.astype(float)
    mask  = ~np.isnan(rain)
    return years[mask], rain[mask]

In [ ]:
def moving_avg(arr, w):
    kernel = np.ones(w) / w
    pad    = w // 2
    padded = np.pad(arr, pad, mode="edge")
    return np.convolve(padded, kernel, mode="valid")[:len(arr)]

# SECTION 3 — NORMAL DISTRIBUTION FITTING  (yearly data)

In [ ]:
def plot_normal_fitting(df, regions, palette, label, filename):
    n     = len(regions)
    ncols = 3
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(14, 3.8 * nrows))   # ← CHANGED: was (18, 5*nrows)
    fig.patch.set_facecolor("#ffffff")
    axes = axes.flatten()

    summary = []

    for idx, (region, color) in enumerate(zip(regions, palette)):
        _, rain   = get_series(df, region)
        ax        = axes[idx]

        mu, sigma     = norm.fit(rain)
        x             = np.linspace(rain.min() - 80, rain.max() + 80, 300)
        ks_stat, ks_p = kstest(rain, "norm", args=(mu, sigma))
        summary.append((region, mu, sigma, ks_stat, ks_p))

        ax.hist(rain, bins="auto", density=True,
                alpha=HIST_ALPHA, color=color, edgecolor="none",
                label="Observed")

        ax.plot(x, norm.pdf(x, mu, sigma),
                color=NORM_COLOR, linewidth=2.0,
                label=f"Normal  μ={mu:.0f}, σ={sigma:.0f}")

        ax.set_xlabel("Annual Rainfall (mm)", fontweight="bold")
        ax.set_ylabel("Probability Density",  fontweight="bold")
        ax.set_title(f"{region}\nKS={ks_stat:.3f}  p={ks_p:.3f}",
                     fontsize=9, fontweight="bold")
        ax.legend(loc="upper right", framealpha=0.7)
        ax.grid(True, alpha=0.25)

    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(                                         
        f"Normal Distribution Fitting — {label}\n"
        "Annual Rainfall  |  2005–2017",
        fontsize=14, fontweight="bold",
        color="#000000",                                 
        y=1.02
    )
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight", facecolor="#ffffff")
    plt.show()
    print(f"\n  [Saved] {filename}")

    print(f"\n{'─'*70}")
    print(f"  Normal Fit Summary — {label}")
    print(f"{'─'*70}")
    print(f"  {'Region':<24}  {'μ (mm)':>9}  {'σ (mm)':>9}  "
          f"{'KS stat':>8}  {'p-value':>8}  Fit?")
    print(f"  {'─'*64}")
    for region, mu, sigma, ks_stat, ks_p in summary:
        fit = "✓ Good" if ks_p > 0.05 else "✗ Poor"
        print(f"  {region:<24}  {mu:>9.1f}  {sigma:>9.1f}  "
              f"{ks_stat:>8.4f}  {ks_p:>8.4f}  {fit}")

    return summary

In [ ]:
# 3A: Normal fitting — Cities 
print("\n" + "═"*70)
print("  SECTION 3A — NORMAL DISTRIBUTION FITTING : CITIES  (yearly)")
print("═"*70)

In [ ]:
norm_cities = plot_normal_fitting(
    yearly_cities_df, CITIES, CITY_PALETTE,
    "Cities", "fig_3A_normal_cities.png"
)

#### INFERENCE — Normal Fitting, Cities (Annual Rainfall)

- **Pune** and **Pimpri Chinchwad** record the highest annual means (>1600 mm)  
  with very large σ (>1300 mm), reflecting extreme orographic amplification  
  from the Western Ghats — the Normal PDF sits awkwardly over a  
  right-skewed histogram.

- **Bengaluru**, **Hyderabad**, and **Hyderabad** occupy the 700–1000 mm mid-range  
  with relatively symmetric histograms, making Normal a reasonable fit.

- **Jaipur** (semi-arid north) has the narrowest spread; its histogram  
  is compact and nearly symmetric — the KS test returns the highest  
  p-value among cities.

- Overall KS p-values are higher for cities with lower and less variable  
  rainfall; for high-rainfall orographic cities the Normal is inadequate  
  and Gamma (Section 4) is preferred.

In [ ]:
# 3B: Normal fitting — States 
print("\n" + "═"*70)
print("  SECTION 3B — NORMAL DISTRIBUTION FITTING : STATES  (yearly)")
print("═"*70)

In [ ]:
norm_states = plot_normal_fitting(
    yearly_states_df, STATES, STATE_PALETTE,
    "States", "fig_3B_normal_states.png"
)

#### INFERENCE — Normal Fitting, States (Annual Rainfall)


- **Meghalaya** dominates with the highest mean (~1400 mm) and large σ;  
  its Normal PDF shows a wide, flat shape that misses the peaked  
  histogram — indicating positive skew.

- **Tamil Nadu** and **Jammu & Kashmir** are the driest JJAS states; their  
  histograms are compact and well-centred, and Normal fits are  
  acceptable (high p-values).

- **Bihar**, **Chhattisgarh**, **Madhya Pradesh**, and **Uttar Pradesh** fall in the  
  750–1100 mm band; moderate σ values and near-symmetric histograms  
  make Normal broadly adequate though not optimal.

- Spatial averaging over large state areas suppresses local extremes,  
  pulling distributions closer to Normal compared with city-scale data.

# SECTION 4 — GAMMA DISTRIBUTION FITTING  (yearly data)

In [ ]:
def plot_gamma_fitting(df, regions, palette, label, filename):
    n     = len(regions)
    ncols = 3
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(14, 3.8 * nrows))   # ← CHANGED: was (18, 5*nrows)
    fig.patch.set_facecolor("#ffffff")
    axes = axes.flatten()

    summary = []

    for idx, (region, color) in enumerate(zip(regions, palette)):
        _, rain           = get_series(df, region)
        ax                = axes[idx]

        shape, loc, scale = gamma.fit(rain, floc=0)
        x                 = np.linspace(0, rain.max() + 100, 300)
        ks_stat, ks_p     = kstest(rain, "gamma", args=(shape, loc, scale))
        summary.append((region, shape, scale, ks_stat, ks_p))

        ax.hist(rain, bins="auto", density=True,
                alpha=HIST_ALPHA, color=color, edgecolor="none",
                label="Observed")

        ax.plot(x, gamma.pdf(x, shape, loc, scale),
                color=GAMMA_COLOR, linewidth=2.0,
                label=f"Gamma  k={shape:.2f}, θ={scale:.0f}")

        ax.set_xlabel("Annual Rainfall (mm)", fontweight="bold")
        ax.set_ylabel("Probability Density",  fontweight="bold")
        ax.set_title(f"{region}\nKS={ks_stat:.3f}  p={ks_p:.3f}",
                     fontsize=9, fontweight="bold")
        ax.legend(loc="upper right", framealpha=0.7)
        ax.grid(True, alpha=0.25)

    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(                                         
        f"Gamma Distribution Fitting — {label}\n"
        "Annual Rainfall  |  2005–2017",
        fontsize=14, fontweight="bold",
        color="#000000",                                  
        y=1.02
    )
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight", facecolor="#ffffff")
    plt.show()
    print(f"\n  [Saved] {filename}")

    print(f"\n{'─'*74}")
    print(f"  Gamma Fit Summary — {label}")
    print(f"{'─'*74}")
    print(f"  {'Region':<24}  {'k (shape)':>10}  {'θ (scale)':>10}  "
          f"{'KS stat':>8}  {'p-value':>8}  Fit?")
    print(f"  {'─'*68}")
    for region, shape, scale, ks_stat, ks_p in summary:
        fit = "✓ Good" if ks_p > 0.05 else "✗ Poor"
        print(f"  {region:<24}  {shape:>10.3f}  {scale:>10.1f}  "
              f"{ks_stat:>8.4f}  {ks_p:>8.4f}  {fit}")

    return summary

In [ ]:
# 4A: Gamma fitting — Cities 
print("\n" + "═"*70)
print("  SECTION 4A — GAMMA DISTRIBUTION FITTING : CITIES  (yearly)")
print("═"*70)

In [ ]:
gamma_cities = plot_gamma_fitting(
    yearly_cities_df, CITIES, CITY_PALETTE,
    "Cities", "fig_4A_gamma_cities.png"
)

#### INFERENCE — Gamma Fitting, Cities (Annual Rainfall)

- **Gamma** is a two-parameter distribution defined for positive values,  
  making it inherently better suited to rainfall than Normal.

- **Pune** and **Pimpri Chinchwad**: *k ≈ 1.5–2* (low shape), indicating a  
  strongly right-skewed distribution — a few extreme wet years dominate  
  the mean; the Gamma PDF traces the histogram tail far more faithfully  
  than the Normal did.

- Cities with *k >> 5* (**Bengaluru**, **Hyderabad**, **Nagpur**) have a nearly  
  bell-shaped Gamma that converges toward Normal — consistent with  
  moderate, less variable annual rainfall.

- **Ahmadabad** and **Jaipur** show intermediate *k*, reflecting semi-arid  
  conditions where occasional wet years create mild positive skew.

- KS p-values are uniformly equal to or higher than the Normal fits,  
  confirming Gamma as the statistically superior parametric choice  
  for city-scale annual rainfall modelling.

In [ ]:
# 4B: Gamma fitting — States 
print("\n" + "═"*70)
print("  SECTION 4B — GAMMA DISTRIBUTION FITTING : STATES  (yearly)")
print("═"*70)

In [ ]:
gamma_states = plot_gamma_fitting(
    yearly_states_df, STATES, STATE_PALETTE,
    "States", "fig_4B_gamma_states.png"
)

#### INFERENCE — Gamma Fitting, States (Annual Rainfall)

- State-level *k* values are generally higher than city-level ones  
  (spatial averaging reduces skewness), placing most states in the  
  *k = 10–50* range where Gamma resembles Normal closely.

- **Meghalaya** retains the lowest *k* among states (~5–8) — its extreme  
  inter-annual variability persists even at the spatial scale of a state.

- **Jammu & Kashmir** and **Tamil Nadu** have the highest *k* values: compact,  
  near-symmetric annual distributions consistent with consistently  
  dry monsoon seasons.

- The high p-values of the Gamma KS test across all states indicate  
  excellent fits — Gamma should be the default distribution for  
  state-level seasonal/annual rainfall in Indian meteorological studies.

# SECTION 5 — DATA SMOOTHING  (yearly data, two kernel widths)

In [ ]:
def plot_smoothing(df, regions, palette, label, filename):
    n     = len(regions)
    ncols = 3
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(14, 3.8 * nrows))   
    fig.patch.set_facecolor("#ffffff")
    axes = axes.flatten()

    for idx, (region, color) in enumerate(zip(regions, palette)):
        years, rain = get_series(df, region)
        ax          = axes[idx]

        ma3 = moving_avg(rain, 3)
        ma5 = moving_avg(rain, 5)

        ax.plot(years, rain,
                color=color, linewidth=1.2, alpha=0.45,
                marker="o", markersize=3.5,             
                label="Raw annual")

        ax.plot(years, ma3,
                color=MA3_COLOR, linewidth=2.0, linestyle="--",
                label="3-yr MA (light)")

        ax.plot(years, ma5,
                color=MA5_COLOR, linewidth=2.0, linestyle="-",
                label="5-yr MA (strong)")

        ax.set_xlabel("Year", fontweight="bold")
        ax.set_ylabel("Annual Rainfall (mm)", fontweight="bold")
        ax.set_title(region, fontsize=9, fontweight="bold")

        ax.set_xticks(years.astype(int))
        ax.tick_params(axis="x", rotation=45, labelsize=7)  
        ax.legend(loc="upper right", framealpha=0.7)
        ax.grid(True, alpha=0.25)

    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(                                            
        f"Data Smoothing — {label}  |  Annual Rainfall  2005–2017\n"
        "Amber dashed = 3-yr MA (light)   Green solid = 5-yr MA (strong)",
        fontsize=13, fontweight="bold",
        color="#000000",                                    
        y=1.02
    )
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight", facecolor="#ffffff")
    plt.show()
    print(f"\n  [Saved] {filename}")

In [ ]:
# 5A: Smoothing — Cities 
print("\n" + "═"*70)
print("  SECTION 5A — DATA SMOOTHING : CITIES  (yearly)")
print("═"*70)

In [ ]:
plot_smoothing(yearly_cities_df, CITIES, CITY_PALETTE,
               "Cities", "fig_5A_smoothing_cities.png")

#### INFERENCE — Data Smoothing, Cities (Annual Rainfall)

- The **3-year moving average (MA)** (amber dashes) removes single-year anomalies while  
  preserving short-term climate signals; the **5-year MA** (green) isolates  
  the broader decadal trend with further noise suppression.

- **Pune** and **Pimpri Chinchwad**: both smoothed lines decline steeply from  
  2005 peaks (~6000 mm) toward a lower plateau by 2012–2017, revealing  
  a structural drying trend over the study window.

- **Nagpur**: the 5-year MA rises gently to a 2012–2013 apex then flattens —  
  no clear long-term decline, unlike the western orographic cities.

- **Bengaluru**: the 5-year MA is nearly horizontal around 900 mm, indicating  
  high year-to-year variability but no directional trend.

- The gap between 3-year and 5-year MAs is widest where abrupt inter-annual  
  swings occur (**Jaipur** 2009 drought, **Ahmadabad** 2008 deficit), marking  
  years of pronounced monsoon failure.

In [ ]:
# 5B: Smoothing — States 
print("\n" + "═"*70)
print("  SECTION 5B — DATA SMOOTHING : STATES  (yearly)")
print("═"*70)

In [ ]:
plot_smoothing(yearly_states_df, STATES, STATE_PALETTE,
               "States", "fig_5B_smoothing_states.png")

#### INFERENCE — Data Smoothing, States (Annual Rainfall)

- State-level smoothed curves are notably tighter (**3-year** and **5-year MAs**  
  nearly overlap) than city-level curves, confirming that spatial  
  averaging dampens local inter-annual noise significantly.

- **Meghalaya**: both MAs show a wave-like pattern peaking near 2005 and  
  2010, reflecting decadal monsoon variability over northeast India.

- **Bihar**: a prominent 2007 wet peak (historic flood season) is visible  
  in the raw series; the 5-year MA smooths it into a gentle dome,  
  suggesting the anomaly was not part of a sustained wet trend.

- **Himachal Pradesh** and **Jammu & Kashmir**: the 5-year MA slopes consistently  
  downward from 2008 onward — a long-term drying signal during JJAS.

- **Chhattisgarh**, **Madhya Pradesh**, **Uttar Pradesh**: moderately declining  
  5-year MA post-2007, consistent with reduced monsoon intensity over the  
  Indo-Gangetic Plain and central India in the study period.

# SECTION 6 — TIME SERIES  

In [ ]:
def plot_timeseries_jjas(df, regions, palette, label, filename):
    fig, ax = plt.subplots(figsize=(13, 5))               
    fig.patch.set_facecolor("#ffffff")

    for region, color in zip(regions, palette):
        years, rain = get_series(df, region)
        ax.plot(years, rain,
                color=color, linewidth=1.8,
                marker="o", markersize=4.5,
                label=region)

    all_years = sorted(df["year"].unique().astype(int))
    ax.set_xticks(all_years)
    ax.set_xticklabels([str(y) for y in all_years],
                       rotation=45, ha="right", fontsize=8, fontweight="bold")
    ax.set_xlabel("Year", fontweight="bold", fontsize=11)
    ax.set_ylabel("Mean JJAS Rainfall (mm)", fontweight="bold", fontsize=11)
    ax.set_title(
        f"JJAS Seasonal Rainfall Time Series — {label}  (2005–2017)",
        fontsize=12, fontweight="bold",
        color="#000000"                                  
    )
    ax.grid(True, alpha=0.25)

    leg = ax.legend(loc="upper right", framealpha=0.85,
                    fontsize=8, ncol=2, edgecolor="#000000")
    for text in leg.get_texts():
        text.set_color("#000000")

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight", facecolor="#ffffff")
    plt.show()
    print(f"\n  [Saved] {filename}")

In [ ]:
# 6A: Time series — Cities (JJAS) 
print("\n" + "═"*70)
print("  SECTION 6A — JJAS TIME SERIES : CITIES")
print("═"*70)

In [ ]:
plot_timeseries_jjas(jjas_cities_df, CITIES, CITY_PALETTE,
                     "Cities", "fig_6A_timeseries_jjas_cities.png")

#### INFERENCE — JJAS Time Series, Cities

- **Pune** and **Pimpri Chinchwad** peak dramatically in 2005 (>6000 mm)  
  then decline sharply — orographic JJAS rainfall at these locations  
  is sensitive to the monsoon's intensity over the Arabian Sea branch.

- The **2009** season is visible as a broad trough across nearly all  
  cities, corresponding to the El Niño–driven all-India deficient  
  monsoon of that year.

- **Nagpur** reaches its highest JJAS value in 2013 (~1480 mm), consistent  
  with the active Bay of Bengal monsoon branch that year.

- **Bengaluru**'s JJAS series oscillates narrowly (~350–670 mm); the city  
  sits partially in a rain-shadow during the southwest monsoon and  
  receives its peak rainfall during the retreating monsoon (Oct–Nov).

- By **2016–2017**, most city JJAS values converge toward a narrower band,  
  suggesting reduced inter-city disparity in recent monsoon seasons.

In [ ]:
# 6B: Time series — States (JJAS) 
print("\n" + "═"*70)
print("  SECTION 6B — JJAS TIME SERIES : STATES")
print("═"*70)

In [ ]:
plot_timeseries_jjas(jjas_states_df, STATES, STATE_PALETTE,
                     "States", "fig_6B_timeseries_jjas_states.png")

#### INFERENCE — JJAS Time Series, States

- **Meghalaya** consistently leads all states (>850–1950 mm JJAS), with  
  a notable peak in 2005 and secondary peak in 2010 — governed by  
  orographic lift of the Bay of Bengal branch over the Khasi Hills.

- **Bihar**'s 2007 peak (~1218 mm) aligns with catastrophic Kosi river  
  floods; state-level JJAS rainfall confirms an exceptionally active  
  Bay of Bengal low-pressure system season that year.

- The **2009** all-India drought imprint is strongest in **Bihar** and  
  **Uttar Pradesh** (largest absolute declines), while **Tamil Nadu**  
  (already dry in JJAS) shows the smallest anomaly.

- **Tamil Nadu** and **Jammu & Kashmir** form a persistent low band  
  (<400–550 mm) throughout the series — structurally different monsoon  
  climates from the rest of the country.

- **Chhattisgarh** and **Madhya Pradesh** are nearly indistinguishable in  
  inter-annual pattern, reflecting their shared central Indian  
  monsoon regime (convergence of Arabian Sea and Bay of Bengal branches).

# SECTION 7 — HYPOTHESIS TESTING  (JJAS seasonal data)

In [ ]:
ALPHA = 0.05

In [ ]:
def run_hypothesis_tests(group_a, group_b, label_a, label_b, context):
    a = np.array(group_a)
    b = np.array(group_b)

    print(f"\n  {'─'*66}")
    print(f"  Hypothesis Test — {context}")
    print(f"  {'─'*66}")
    print(f"  H₀ : μ({label_a}) = μ({label_b})")
    print(f"  H₁ : μ({label_a}) ≠ μ({label_b})")
    print(f"  Significance level  α = {ALPHA}  (two-tailed)")
    print()
    print(f"  Group A — {label_a}")
    print(f"    n        = {len(a)}")
    print(f"    Mean     = {np.mean(a):.2f} mm")
    print(f"    Std Dev  = {np.std(a, ddof=1):.2f} mm")
    print(f"    Range    = [{np.min(a):.1f},  {np.max(a):.1f}] mm")
    print()
    print(f"  Group B — {label_b}")
    print(f"    n        = {len(b)}")
    print(f"    Mean     = {np.mean(b):.2f} mm")
    print(f"    Std Dev  = {np.std(b, ddof=1):.2f} mm")
    print(f"    Range    = [{np.min(b):.1f},  {np.max(b):.1f}] mm")

    t_stat, t_p = ttest_ind(a, b, equal_var=False)
    print(f"\n  ── Welch's t-test (parametric) ───────────────────────────────")
    print(f"     Assumes: independent samples; does NOT assume equal variances.")
    print(f"     t-statistic = {t_stat:.4f}")
    print(f"     p-value     = {t_p:.6f}")
    if t_p < ALPHA:
        print(f"     Decision  → REJECT H₀   (p = {t_p:.6f} < α = {ALPHA})")
        print(f"     Conclusion: The difference in means is statistically significant.")
    else:
        print(f"     Decision  → FAIL TO REJECT H₀   (p = {t_p:.6f} ≥ α = {ALPHA})")
        print(f"     Conclusion: No statistically significant difference detected.")

    u_stat, u_p = mannwhitneyu(a, b, alternative="two-sided")
    print(f"\n  ── Mann-Whitney U test (non-parametric, distribution-free) ────")
    print(f"     Assumes: independent samples; no distributional assumptions.")
    print(f"     U-statistic = {u_stat:.1f}")
    print(f"     p-value     = {u_p:.6f}")
    if u_p < ALPHA:
        print(f"     Decision  → REJECT H₀   (p = {u_p:.6f} < α = {ALPHA})")
        print(f"     Conclusion: The rank-based distributions differ significantly.")
    else:
        print(f"     Decision  → FAIL TO REJECT H₀   (p = {u_p:.6f} ≥ α = {ALPHA})")
        print(f"     Conclusion: No significant distributional difference detected.")

    both_reject = (t_p < ALPHA) and (u_p < ALPHA)
    print(f"\n  ── Joint Conclusion ──────────────────────────────────────────")
    if both_reject:
        print(f"     Both tests reject H₀.")
        print(f"     → Strong statistical evidence that {label_a}")
        print(f"       and {label_b} have DIFFERENT mean JJAS rainfall.")
    elif (t_p < ALPHA) or (u_p < ALPHA):
        print(f"     Mixed result — one test rejects H₀, the other does not.")
        print(f"     → Interpret with caution; check sample size and skewness.")
    else:
        print(f"     Both tests fail to reject H₀.")
        print(f"     → No significant difference between {label_a}")
        print(f"       and {label_b} in mean JJAS rainfall.")

    return t_stat, t_p, u_stat, u_p

In [ ]:
# 7A: Hypothesis testing — Cities 
print("\n" + "═"*70)
print("  SECTION 7A — HYPOTHESIS TESTING : CITIES  (JJAS seasonal data)")
print("═"*70)
print("""
  Group definitions:
  ─────────────────────────────────────────────────────────────────
  Group A — High-Rainfall Cities : Nagpur, Pune, Pimpri Chinchwad, Indore
  Group B — Low-Rainfall Cities  : Ahmadabad, Bengaluru, Hyderabad, Jaipur
""")

In [ ]:
high_city_regions = ["NAGPUR", "PUNE", "PIMPRI CHINCHWAD", "INDORE"]
low_city_regions  = ["AHMADABAD", "BENGALURU", "HYDERABAD", "JAIPUR"]

In [ ]:
high_city_data = np.concatenate([
    get_series(jjas_cities_df, r)[1] for r in high_city_regions
])
low_city_data = np.concatenate([
    get_series(jjas_cities_df, r)[1] for r in low_city_regions
])

In [ ]:
run_hypothesis_tests(
    high_city_data, low_city_data,
    "High-Rainfall Cities", "Low-Rainfall Cities",
    "JJAS Seasonal Rainfall  2005–2017"
)

#### DETAILED INTERPRETATION — Hypothesis Testing, Cities (JJAS)

---

- **Welch’s t-test** is used because the two groups have markedly unequal  
  variances — **Pune / Pimpri Chinchwad** JJAS σ is very large while  
  **Bengaluru / Jaipur** JJAS σ is compact.

- The **Mann–Whitney U test** provides a non-parametric check avoiding  
  distributional assumptions entirely — appropriate since JJAS rainfall  
  is right-skewed for orographic cities.

- If both tests reject *H₀* at *α = 0.05*: the difference in monsoon-season  
  rainfall is a real, persistent climatic signal, not sampling noise.

- **Physical explanation:** During JJAS, **Pune** and **Pimpri Chinchwad**  
  intercept the full force of the Arabian Sea monsoon branch over the  
  Western Ghats (windward orographic effect), while **Jaipur** and  
  **Ahmadabad** sit in the Thar Desert fringe far from oceanic moisture  
  sources — producing JJAS rainfall disparities of **3–8×**.

In [ ]:
# 7B: Hypothesis testing — States 
print("\n" + "═"*70)
print("  SECTION 7B — HYPOTHESIS TESTING : STATES  (JJAS seasonal data)")
print("═"*70)
print("""
  Group definitions:
  ─────────────────────────────────────────────────────────────────
  Group A — High-Rainfall States : Meghalaya, Chhattisgarh, Bihar
  Group B — Low-Rainfall States  : Tamil Nadu, Jammu & Kashmir, Himachal Pradesh
""")

In [ ]:
high_state_regions = ["MEGHALAYA", "CHHATTISGARH", "BIHAR"]
low_state_regions  = ["TAMIL NADU", "JAMMU & KASHMIR", "HIMACHAL PRADESH"]

In [ ]:
high_state_data = np.concatenate([
    get_series(jjas_states_df, r)[1] for r in high_state_regions
])
low_state_data = np.concatenate([
    get_series(jjas_states_df, r)[1] for r in low_state_regions
])

In [ ]:
run_hypothesis_tests(
    high_state_data, low_state_data,
    "High-Rainfall States", "Low-Rainfall States",
    "JJAS Seasonal Rainfall  2005–2017"
)

#### DETAILED INTERPRETATION — Hypothesis Testing, States (JJAS)

- The mean JJAS difference between high-rain and low-rain state groups  
  is large and consistent across all 13 monsoon seasons — making  
  statistical significance highly likely.

- **Welch’s t-test** accounts for **Meghalaya**’s far larger JJAS variance  
  versus **Tamil Nadu**’s compact JJAS distribution (Tamil Nadu’s primary  
  season is the Northeast Monsoon, Oct–Dec, not JJAS).

- The **Mann–Whitney U test** provides independent rank-based confirmation:  
  if **Meghalaya**’s lowest JJAS year still exceeds **Tamil Nadu**’s highest,  
  the U-statistic will be extreme and p-value near zero.

- Joint rejection of *H₀* by both tests is strong evidence that the  
  JJAS rainfall disparity is a fundamental feature of India’s monsoon  
  climate geography, not a statistical artefact.

- **Policy implication:** Flood management (**Meghalaya**, **Bihar**) and  
  drought resilience planning (**Tamil Nadu**, **Jammu & Kashmir**) must be  
  anchored in this statistically confirmed, physically grounded JJAS disparity.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═"*70)
print("  ANALYSIS COMPLETE — 8 figures saved.")
print("═"*70)